# EXP08: 资金管理 + MaxPos + Cooldown 综合扫描

**合并**: NB20

本金 $3,000 场景下，测试不同 RISK_PER_TRADE / MAX_LOT_SIZE / MaxPositions / Cooldown

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, C12_KWARGS
from backtest.stats import print_ranked
import config

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}
print('Data loaded')

## Part 1: MaxPositions 扫描

In [ ]:
mp_results = []
for mp in [1, 2, 3]:
    r = run_variant(data, f"MaxPos={mp}", **{**LIVE_KWARGS, "max_positions": mp})
    r['max_pos'] = mp
    mp_results.append(r)
print_ranked(mp_results)

## Part 2: Cooldown 精细扫描

In [ ]:
cd_results = []
for cd in [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]:
    r = run_variant(data, f"CD={cd}h", **{**LIVE_KWARGS, "cooldown_hours": cd})
    r['cooldown_h'] = cd
    cd_results.append(r)
print_ranked(cd_results)

## Part 3: RISK_PER_TRADE 扫描 (Capital=$3000)

In [ ]:
original_rpt = config.RISK_PER_TRADE
original_max_lot = config.MAX_LOT_SIZE

risk_results = []
for rpt, max_lot, label in [
    (50, 0.03, "$50/trade_0.03lot (current)"),
    (50, 0.05, "$50/trade_0.05lot"),
    (75, 0.03, "$75/trade_0.03lot"),
    (75, 0.05, "$75/trade_0.05lot"),
    (90, 0.05, "$90/trade_0.05lot"),
    (60, 0.04, "$60/trade_0.04lot"),
]:
    config.RISK_PER_TRADE = rpt
    config.MAX_LOT_SIZE = max_lot
    r = run_variant(data, label, **LIVE_KWARGS)
    r['risk_per_trade'] = rpt
    r['max_lot'] = max_lot
    risk_results.append(r)

config.RISK_PER_TRADE = original_rpt
config.MAX_LOT_SIZE = original_max_lot
print_ranked(risk_results)

## Part 4: MaxDD 占比分析

In [ ]:
print("=== MaxDD as % of $3000 Capital ===")
for r in risk_results:
    max_dd = r.get('max_dd', 0)
    dd_pct = abs(max_dd) / 3000 * 100
    print(f"  {r['label']:35s}: MaxDD=${abs(max_dd):7.0f} ({dd_pct:5.1f}% of $3000), "
          f"Sharpe={r['sharpe']:.2f}, PnL=${r['total_pnl']:.0f}")

In [ ]:
import json
from backtest.runner import sanitize_for_json
all_results = mp_results + cd_results + risk_results
with open('../data/exp08_results.json', 'w') as f:
    json.dump(sanitize_for_json(all_results), f, indent=2)
print('Saved')